In [ ]:
import os
print(os.listdir())

['.config', 'duka_sales.csv', 'drive', 'sample_data']


In [ ]:
import pandas as pd
import numpy as np

sales = pd.read_csv("duka_sales.csv")

sales["order_date"] = pd.to_datetime(
    sales["order_date"]
)

sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


# Duka Electronics Sales Exploration

# Executive Summary

- The dataset contains 1,000 sales orders covering January to June 2026.
- Nairobi receives the highest proportion of orders because it accounts for about 40% of generated transactions.
- Premium products priced at KES 28,000 and above contribute substantially to revenue.
- Sales performance differs across branches, products and channels.
- Revenue rankings and order-count rankings are not always identical because products have different prices.

In [ ]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]

sales["month"] = sales["order_date"].dt.month_name()

sales["weekday"] = sales["order_date"].dt.day_name()

sales["is_weekend"] = sales["weekday"].isin(
    ["Saturday", "Sunday"]
)

sales["price_band"] = np.where(
    sales["unit_price"] >= 28000,
    "Premium",
    "Value"
)

sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000,45000,June,Wednesday,False,Value
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000,260000,February,Monday,False,Premium
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000,140000,January,Monday,False,Premium
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500,12500,April,Sunday,True,Value
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500,7500,March,Saturday,True,Value


In [ ]:
sales.columns

Index(['order_id', 'order_date', 'branch', 'product', 'quantity', 'channel',
       'unit_price', 'revenue', 'month', 'weekday', 'is_weekend',
       'price_band'],
      dtype='object')

# Task 3: Branch Ranking

This task ranks branches by total revenue and total orders to compare business performance across locations.

In [ ]:
branch_revenue = (
    sales.groupby("branch")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

branch_orders = (
    sales.groupby("branch")["order_id"]
    .count()
    .sort_values(ascending=False)
)

print("Revenue Ranking")
print(branch_revenue)

print("\nOrder Ranking")
print(branch_orders)

Revenue Ranking
branch
Nairobi    26106100
Mombasa    14003700
Nakuru     11970400
Kisumu     10868100
Eldoret     8844200
Name: revenue, dtype: int64

Order Ranking
branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: order_id, dtype: int64


## Task 3 Findings

The revenue ranking and order ranking are not necessarily identical. Revenue depends on both the number of orders and the values of products sold. A branch with fewer orders can generate more revenue if it sells more high-value products.

# Task 4: Top Products by Revenue and Units Sold

This task identifies the products generating the most revenue and the products selling the highest number of units.

In [ ]:
top_revenue_products = (
    sales.groupby("product")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

top_units_products = (
    sales.groupby("product")["quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(3)
)

print("Top 3 Products by Revenue")
print(top_revenue_products)

print("\nTop 3 Products by Units Sold")
print(top_units_products)


Top 3 Products by Revenue
product
Laptop    41535000
Tablet    19348000
Phone      8880000
Name: revenue, dtype: int64

Top 3 Products by Units Sold
product
Tablet    691
Laptop    639
Phone     592
Name: quantity, dtype: int64


## Task 4 Findings

Products contributing the highest revenue may differ from those selling the highest number of units because products vary significantly in price. Premium products can generate more revenue even with fewer sales.

# Task 5: Weekend vs Weekday Analysis

This task compares the number of orders, total revenue and average order revenue between weekends and weekdays.

In [ ]:
weekend_analysis = sales.groupby("is_weekend").agg(
    orders=("order_id", "count"),
    total_revenue=("revenue", "sum"),
    average_order_revenue=("revenue", "mean")
)

weekend_analysis

,orders,total_revenue,average_order_revenue
is_weekend,,,
False,723,51856200,71723.651452
True,277,19936300,71972.202166


## Task 5 Findings

This comparison helps identify differences in customer activity between weekdays and weekends. The results can guide scheduling, staffing and promotional strategies.

# Task 6: Monthly Revenue Analysis

This task determines the best and worst performing months by revenue and calculates the percentage difference.

In [ ]:
monthly_revenue = sales.groupby("month")["revenue"].sum()

best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()

best_revenue = monthly_revenue.max()
worst_revenue = monthly_revenue.min()

percentage_difference = (
    (best_revenue - worst_revenue)
    / worst_revenue
) * 100

print("Best Month:", best_month)
print("Worst Month:", worst_month)

print(f"Revenue Difference: {percentage_difference:.2f}%")

Best Month: April
Worst Month: June
Revenue Difference: 23.98%


In [ ]:
branch_revenue = (
    sales.groupby("branch")["revenue"]
    .sum()
    .sort_values(ascending=False)
)

branch_revenue

,revenue
branch,
Nairobi,26106100
Mombasa,14003700
Nakuru,11970400
Kisumu,10868100
Eldoret,8844200


In [ ]:
pivot = pd.pivot_table(
    sales,
    values="revenue",
    index="branch",
    columns="channel",
    aggfunc="sum"
)

pivot

channel,In-store,Online,Phone order
branch,,,
Eldoret,4885300,3450700,508200
Kisumu,6468600,3370800,1028700
Mombasa,8039600,4086800,1877300
Nairobi,14808400,7527400,3770300
Nakuru,6767300,4350400,852700


## Task 6 Findings

The best-performing month generated significantly more revenue than the lowest-performing month. This difference highlights seasonal trends that may influence future sales planning.

# Task 7: Premium Online Orders from Nairobi and Mombasa

This task filters Premium Online orders from Nairobi and Mombasa and exports the results to a CSV file.

In [ ]:
premium_orders = sales[
    (sales["price_band"] == "Premium")
    & (sales["channel"] == "Online")
    & (sales["branch"].isin(["Nairobi", "Mombasa"]))
]

print("Number of Orders:")
print(len(premium_orders))

print("\nTotal Revenue:")
print(f"KES {premium_orders['revenue'].sum():,.0f}")

Number of Orders:
83

Total Revenue:
KES 10,218,000


In [ ]:
premium_orders.to_csv(
    "premium_online_coast_capital.csv",
    index=False
)

print("Export completed successfully.")

Export completed successfully.


## Task 7 Findings

The filtered dataset contains Premium Online orders originating from Nairobi and Mombasa only. These transactions represent high-value digital sales and can be used for targeted market analysis.

In [ ]:
sales.columns

Index(['order_id', 'order_date', 'branch', 'product', 'quantity', 'channel',
       'unit_price', 'revenue', 'month', 'weekday', 'is_weekend',
       'price_band'],
      dtype='object')

# Executive Summary

- The dataset contains 1,000 sales orders covering January to June 2026.
- Nairobi generated the highest volume of transactions among all branches.
- Premium products contributed a significant portion of total revenue despite representing fewer products.
- Online and weekend sales patterns reveal opportunities for targeted marketing initiatives.
- Revenue performance varied substantially across branches, products and months.